In [1]:
!pip install anndata==0.8.0

In [35]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from sklearn.neighbors import KernelDensity
import time
import dill
import pickle
from scipy.spatial import distance
import os
import scipy

In [36]:
with open('../../parent_dict_v2.pkl', 'rb') as f:
    parent_dict = pickle.load(f)

In [37]:
fn = '../../Active_SAM_joined/SAM_MO_soupx_plus5_cleaned_03122025.h5ad'

In [38]:
sam = SAM()
sam.load_data(fn)

In [80]:
lsaturn_m = pd.read_csv('../../SATURN_hypo_september2026_mapping_30_seeds/MO_30_seeds.csv', index_col = 'barcode')

In [81]:
meta_folder = '../../Active_SAMap_Joined/MO_metadata/'

In [82]:
mn = os.listdir('../../Active_SAMap_Joined/MO_metadata/')

In [83]:
pd.read_csv(meta_folder + mn[3])

,Unnamed: 0,orig.ident,nCount_RNA,nFeature_RNA,n_counts,n_genes,key,eq_subclass_nounlabeled,leiden_removal,SCT_snn_res.5,leiden_clusters,subclass_id_label_mapping,subclass_id_label_lc
0,AAACCCATCTCAAAGC,MO,11772.0,3844,11772.0,3844,Run12_sample1,mo_9,35,15,86,Unlabeled,86
1,AAACGAACAGCACGAA,MO,4877.0,2561,4877.0,2561,Run12_sample1,mo_5,39,105,240,Unlabeled,240
2,AAACGAAGTCGGCACT,MO,5221.0,2859,5221.0,2859,Run12_sample1,108 ARH-PVp Tbx3 Gaba,35,14,16,108 ARH-PVp Tbx3 Gaba,16
3,AAACGAATCCACCCTA,MO,7922.0,3360,7922.0,3360,Run12_sample1,mo_2,78,87,527,Unlabeled,527
4,AAACGAATCCCAGCGA,MO,7020.0,3204,7020.0,3204,Run12_sample1,128 VMH Fezf1 Glut,2,16,333,128 VMH Fezf1 Glut,333
...,...,...,...,...,...,...,...,...,...,...,...,...,...
75325,TTTGTTGGTACTAACC,MO,5496.0,2569,5496.0,2569,Run12_sample12,057 NDB-SI-MA-STRv Lhx8 Gaba,8,26,156,057 NDB-SI-MA-STRv Lhx8 Gaba,156
75326,TTTGTTGGTCACAATC-1,MO,2974.0,1804,2974.0,1804,Run12_sample12,089 PVR Six3 Sox3 Gaba,11,52,119,089 PVR Six3 Sox3 Gaba,119
75327,TTTGTTGTCCAACCAA,MO,5601.0,2608,5601.0,2608,Run12_sample12,mo_9,5,56,265,Unlabeled,265
75328,TTTGTTGTCCCTGTTG,MO,6209.0,2759,6209.0,2759,Run12_sample12,mo_3,1,33,522,Unlabeled,522


In [84]:
test = pd.read_csv(meta_folder + mn[0])['subclass_id_label_mapping']

In [85]:
barcodes = pd.read_csv(meta_folder + mn[0])['Unnamed: 0']

In [86]:
raw_lc = pd.read_csv(meta_folder + mn[0])['subclass_id_label_lc']

In [87]:
df_lc = pd.DataFrame(data = list(raw_lc), index = list(barcodes), columns = ['raw_lc'])

In [88]:
mode_df = pd.DataFrame(index = [a for a in range(len(test))], columns = [b for b in range(len(mn))])
for j in range(len(mn)):
    print(mn[j])
    dat = list(pd.read_csv(meta_folder + mn[j])['subclass_id_label_lc'])
    mode_df.loc[:,j] = dat

MO_metadata_soupxplus5_cleaned_03122025_subclass_250_22.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_8.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_17.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_11.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_18.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_19.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_14.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_15.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_7.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_25.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_28.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_29.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_16.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_13.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_24.csv
MO_metadata_soupxplus5_cleaned_03122025_subclass_250_21.csv
MO_metadata_soupxplus5_cleaned_03122025_su

In [89]:
fin = []
for item in mode_df.columns:
    fin.append(mode_df.loc[10000,item])

In [90]:
scipy.stats.mode(fin)

ModeResult(mode=array([62]), count=array([30]))

In [91]:
import re

mn = sorted(mn, key=lambda s: [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)])

In [92]:
mn

['MO_metadata_soupxplus5_cleaned_03122025_subclass_250_0.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_1.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_2.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_3.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_4.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_5.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_6.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_7.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_8.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_9.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_10.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_11.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_12.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_13.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass_250_14.csv',
 'MO_metadata_soupxplus5_cleaned_03122025_subclass

In [93]:
lsamap_mv = pd.DataFrame(index = [a for a in barcodes], columns = [b for b in range(len(mn))])
for j in range(len(mn)):
    dat = list(pd.read_csv(meta_folder + mn[j])['subclass_id_label_mapping'])
    lsamap_mv.loc[:,j] = dat

In [94]:
lsamap_m = lsamap_mv

In [ ]:
#for vole
threshold = .75
mapping_dict ={}
a = 0
b = 0
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    a += 1
    inp = ['Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    if  f_saturn_m[0]/len(ct_saturn_m)>= threshold:
        inp[0] = mode_saturn_m[0]
    if f_samap_m[0]/len(ct_samap_m) >= threshold:
        inp[1] = mode_samap_m[0]
    if inp[1] != inp[0] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
        if inp[0] == '077 CEA-BST Gal Avp Gaba':
            mapping_dict[lc] = 'mg_077_106'
        elif inp[0] == '105 TMd-DMH Foxd2 Gaba':
            mapping_dict[lc] = 'mg_105_107'
        print(inp[0],inp[1])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [96]:
new_mapping = []
new_mapping_name = 'test'
for item in sam.adata.obs['eq_subclass_lc']:
    new_mapping.append(mapping_dict[item])
sam.adata.obs[new_mapping_name] = new_mapping

In [76]:
np.sort(sam.adata.obs['ss_subclass'].unique())

array(['007 L2/3 IT CTX Glut', '009 L2/3 IT PIR-ENTl Glut',
       '010 IT AON-TT-DP Glut', '014 LA-BLA-BMA-PA Glut',
       '016 CA1-ProS Glut', '017 CA3 Glut', '022 L5 ET CTX Glut',
       '037 DG Glut', '041 OB-in Frmd7 Gaba', '047 Sncg Gaba',
       '049 Lamp5 Gaba', '054 STR Prox1 Lhx6 Gaba', '055 STR Lhx8 Gaba',
       '056 Sst Chodl Gaba', '057 NDB-SI-MA-STRv Lhx8 Gaba',
       '058 PAL-STR Gaba-Chol', '059 GPe-SI Sox6 Cyp26b1 Gaba',
       '061 STR D1 Gaba', '062 STR D2 Gaba', '063 STR D1 Sema5a Gaba',
       '064 STR-PAL Chst9 Gaba', '066 NDB-SI-ant Prdm12 Gaba',
       '067 LSX Sall3 Pax6 Gaba', '068 LSX Otx2 Gaba',
       '069 LSX Nkx2-1 Gaba', '070 LSX Prdm12 Slit2 Gaba',
       '073 MEA-BST Sox6 Gaba', '074 MEA-BST Lhx6 Sp9 Gaba',
       '075 MEA-BST Lhx6 Nr2e1 Gaba', '076 MEA-BST Lhx6 Nfib Gaba',
       '077 CEA-BST Gal Avp Gaba', '078 SI-MA-ACB Ebf1 Bnc2 Gaba',
       '079 CEA-BST Six3 Cyp26b1 Gaba', '080 CEA-AAA-BST Six3 Sp9 Gaba',
       '081 ACB-BST-FS D1 Gaba', '082 

In [29]:
a = 0
for item in sam.adata.obs_names:
    if df.loc[item,'fin_ct'] == sam.adata.obs.loc[item,'test']:
        print(df.loc[item,'fin_ct'],sam.adata.obs.loc[item,'test'])
        a += 1

Unlabeled Unlabeled


In [28]:
a

75330

In [27]:
sam.adata.obs

,orig.ident,nCount_RNA,nFeature_RNA,n_counts,n_genes,key,subclass_id_label_mapping,subclass_id_label_lc,leiden_clusters,subclass_id_label_mapping_nounlabeled,...,NN,ss_subclass,ss_subclass_nounlabeled,ss_subclass_nounlabeled_03102026,ss_subclass_nounlabeled_03102026_crossed,neurotransmitter_v2,test,ss_subclass_threshold_90,yanay_labels,yanay_label_nounlabeled
AAACCCATCTCAAAGC,MO,11772.0,3844,11772.0,3844,Run12_sample1,Unlabeled,131,32,mo_15,...,Unlabeled,Unlabeled,mo_18,mo_5,107 DMH Hmx2 Gaba,Gaba,Unlabeled,Unlabeled,Unlabeled,mo_28
AAACGAACAGCACGAA,MO,4877.0,2561,4877.0,2561,Run12_sample1,Unlabeled,290,24,mo_10,...,Unlabeled,Unlabeled,mo_6,mo_0,124 MPN-MPO-PVpo Hmx2 Glut,Glut,Unlabeled,Unlabeled,Unlabeled,mo_8
AAACGAAGTCGGCACT,MO,5221.0,2859,5221.0,2859,Run12_sample1,108 ARH-PVp Tbx3 Gaba,27,12,108 ARH-PVp Tbx3 Gaba,...,Neuron,108 ARH-PVp Tbx3 Gaba,108 ARH-PVp Tbx3 Gaba,108 ARH-PVp Tbx3 Gaba,108 ARH-PVp Tbx3 Gaba,Gaba,108 ARH-PVp Tbx3 Gaba,108 ARH-PVp Tbx3 Gaba,108 ARH-PVp Tbx3 Gaba,108 ARH-PVp Tbx3 Gaba
AAACGAATCCACCCTA,MO,7922.0,3360,7922.0,3360,Run12_sample1,Unlabeled,529,85,mo_9,...,Unlabeled,Unlabeled,mo_38,mo_23,133 PVH-SO-PVa Otp Glut,Glut,Unlabeled,Unlabeled,Unlabeled,mo_47
AAACGAATCCCAGCGA,MO,7020.0,3204,7020.0,3204,Run12_sample1,128 VMH Fezf1 Glut,104,31,128 VMH Fezf1 Glut,...,Neuron,128 VMH Fezf1 Glut,128 VMH Fezf1 Glut,128 VMH Fezf1 Glut,128 VMH Fezf1 Glut,Glut,128 VMH Fezf1 Glut,128 VMH Fezf1 Glut,128 VMH Fezf1 Glut,128 VMH Fezf1 Glut
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGGTACTAACC,MO,5496.0,2569,5496.0,2569,Run12_sample12,057 NDB-SI-MA-STRv Lhx8 Gaba,217,5,057 NDB-SI-MA-STRv Lhx8 Gaba,...,Neuron,057 NDB-SI-MA-STRv Lhx8 Gaba,057 NDB-SI-MA-STRv Lhx8 Gaba,057 NDB-SI-MA-STRv Lhx8 Gaba,057 NDB-SI-MA-STRv Lhx8 Gaba,Gaba,057 NDB-SI-MA-STRv Lhx8 Gaba,057 NDB-SI-MA-STRv Lhx8 Gaba,057 NDB-SI-MA-STRv Lhx8 Gaba,057 NDB-SI-MA-STRv Lhx8 Gaba
TTTGTTGGTCACAATC-1,MO,2974.0,1804,2974.0,1804,Run12_sample12,089 PVR Six3 Sox3 Gaba,204,13,089 PVR Six3 Sox3 Gaba,...,Neuron,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba
TTTGTTGTCCAACCAA,MO,5601.0,2608,5601.0,2608,Run12_sample12,Unlabeled,211,13,mo_11,...,Unlabeled,Unlabeled,mo_4,mo_5,107 DMH Hmx2 Gaba,Gaba,Unlabeled,Unlabeled,Unlabeled,mo_7
TTTGTTGTCCCTGTTG,MO,6209.0,2759,6209.0,2759,Run12_sample12,Unlabeled,478,8,mo_4,...,Unlabeled,Unlabeled,mo_7,mo_1,mo_1,Glut,Unlabeled,Unlabeled,Unlabeled,mo_11


In [100]:
a = 0
fin_obs = []
for item in sam.adata.obs_names:
    if sam.adata.obs.loc[item,'test'] == sam.adata.obs.loc[item,'ss_subclass']:
        a += 1
        fin_obs.append(item)

In [101]:
a

75330

In [102]:
len(sam.adata)

75330

In [58]:
sam.save_anndata(fn)